In [2]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [3]:
# === Настройки ===
# Корневая папка, где лежат папки показателей с подпапкой results/
ROOT = Path(".")  # при необходимости поменяй
# Маска для файлов с 1-месячным прогнозом (как ты сохраняешь)
GLOB_PATTERN = "**/results/* - Прогноз на 2025-08 (как в обучении).xlsx"

# Сопоставление названий столбцов в файлах результата к целевым под-колонкам
COL_MAP = {
    "Факт (2025-08)": "Фактическое значение",
    "Прогноз (2025-08)": "Прогнозное значение",
    "Отклонение, %": "% отклонение",
}

# === Сбор всех файлов результатов по показателям ===
files = list(ROOT.glob(GLOB_PATTERN))
if not files:
    raise FileNotFoundError("Не найдено ни одного файла результатов по маске: " + GLOB_PATTERN)

wide_parts = []
all_regions = set()

for fp in files:
    # Имя показателя берём из имени файла (до " - Прогноз ...")
    # Пример: "КРС - Прогноз на 2025-08 (как в обучении).xlsx" -> "КРС"
    indicator = fp.stem.split(" - ")[0].strip()

    df_res = pd.read_excel(fp)

    # Ожидаемые колонки: Регион, Лучший метод, Прогноз (2025-08), Факт (2025-08), Отклонение, %
    # Берём только нужные 3 метрики + регион
    must_have = ["Регион"] + list(COL_MAP.keys())
    missing = [c for c in must_have if c not in df_res.columns]
    if missing:
        raise ValueError(f"В файле '{fp}' отсутствуют нужные колонки: {missing}")

    sub = df_res[["Регион"] + list(COL_MAP.keys())].copy()

    # Приводим к числам на всякий случай
    for c in COL_MAP.keys():
        sub[c] = pd.to_numeric(sub[c], errors="coerce")

    # Сохраним список регионов
    all_regions.update(sub["Регион"].dropna().unique())

    # Готовим MultiIndex-колонки: (Индикатор, Подколонка)
    sub = sub.set_index("Регион")
    sub.columns = pd.MultiIndex.from_product([[indicator],
                                              [COL_MAP[c] for c in sub.columns]])

    wide_parts.append(sub)

# === Объединяем по регионам (outer join), чтобы учесть регистры всех файлов ===
if not wide_parts:
    raise RuntimeError("Список частей пуст — проверь папки и имена файлов.")

# Полный список регионов
regions_order = sorted(all_regions)

# Стартуем с пустого DataFrame с нужным индексом
wide = pd.DataFrame(index=regions_order)

# Последовательно добавляем блоки по показателям
for part in wide_parts:
    # убедимся, что индекс — регионы, добавим недостающие как NaN
    part = part.reindex(regions_order)
    wide = wide.join(part, how="outer")

# Переносим индекс в колонку
wide = wide.reset_index().rename(columns={"index": "Регион"})

# === Сохранение результата ===
out_path = ROOT / "Все показатели - Прогноз на 2025-08 (сводная).xlsx"
out_path.parent.mkdir(parents=True, exist_ok=True)
wide.to_excel(out_path, index=False)

print(f"Готово. Сводная таблица сохранена: {out_path}")


Готово. Сводная таблица сохранена: Все показатели - Прогноз на 2025-08 (сводная).xlsx


In [ ]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по яйцам v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



In [ ]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Яйца - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

In [ ]:
actual_aug = pd.read_excel("Яйца 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Яйца"] = (actual_aug["Яйца"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Яйца обработанные август 2025.xlsx", index=False)
actual_aug



In [ ]:
# === настройки ===
TARGET = "Яйца"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [ ]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [ ]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [ ]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [ ]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [ ]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [ ]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Яйца - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug